# CH12 project 1 — the camera on the screen

The MIPI camera into the PL, out to DDR, and onto the DisplayPort. No
accelerator: this notebook is about getting a picture at all, and about the two
facts every later project depends on — **what byte order camera frames arrive
in**, and **what frame rate the path sustains before any filtering is added**.

The sensor is an **OV5647** — the one in a Raspberry Pi Camera Module v1 — not
the Pcam 5C this chapter was first written around. It sits at I2C address 0x36
instead of 0x3c, and its register map is a different map, so PYNQ's `Pcam5C`
class and the `libpcam5c.so` it wraps cannot drive it. CH12 brings its own
driver instead, in `sw/ov5647.py`.

That turned out to be worth more than it cost. `libpcam5c.so` did two jobs at
once — the sensor over I2C, *and* the demosaic, gamma-LUT and colour-space
blocks in the PL — and it did them behind a closed door. Replacing it means
those blocks are now programmed by code you can read, so this notebook can hand
you gamma, saturation, brightness and exposure as live controls. The previous
version of this notebook had a section apologising that it could not.

Run it as root, with `XILINX_XRT` set, from the directory holding
`camera_dp.bit`, `camera_dp.hwh` and `camera_dp.dtbo`:

```
sudo env XILINX_XRT=/usr /usr/local/share/pynq-venv/bin/jupyter notebook
```

Root is needed to program the PL and to open the DRM device, and `XILINX_XRT`
because `/etc/profile.d/xrt_setup.sh` does not run for non-interactive shells.

## Load the overlay

In [ ]:
import pathlib
import sys
import time

import numpy as np
import PIL.Image
from IPython.display import display

from pynq import Overlay
from pynq.lib.video import VideoMode, DisplayPort, PIXEL_RGB

# CH12's own camera driver. Importing it is what binds `ol.mipi` to it later,
# so it has to happen BEFORE the Overlay is constructed -- PYNQ resolves
# hierarchy drivers newest-registered-first, and `Pcam5C` is already
# registered by `pynq.lib.video` above.
for candidate in ("../../sw", "../sw", "."):
    if (pathlib.Path(candidate) / "ov5647.py").exists():
        sys.path.insert(0, candidate)
        break
import ov5647 as cam

W, H = 1280, 720

def find_bitstream(name, variant=None):
    """Locate a bitstream in the repo layout or beside this notebook.

    In the repo the builds land in ../out_<variant>/; on the board everything
    is normally copied into one directory. Try both rather than making the
    reader edit a path.
    """
    cands = []
    if variant:
        cands.append(f"../out_{variant}/{name}")
    cands += [f"../out/{name}", name]
    for c in cands:
        if pathlib.Path(c).exists():
            return c
    raise FileNotFoundError(f"{name} not found -- looked in {cands}")

BITSTREAM = find_bitstream("camera_dp.bit")
print("overlay:", BITSTREAM)
ol = Overlay(BITSTREAM)
list(ol.ip_dict.keys())

`camera_dp.dtbo` must sit next to `camera_dp.bit` — PYNQ matches an overlay to a
bitstream by basename, and without it the AXI IIC controller never appears in
`/dev/i2c-*` and the camera cannot be talked to at all.

## First, release the video IP reset

Before anything reads a register belonging to `demosaic`, `gamma_lut`,
`v_proc_sys`, `axis_channel_swap` or `pixel_pack`. This is not a nicety; get it
wrong and the board is gone.

`gpio_ip_reset` has two channels. Channel 2 is the camera module's own reset and
the hierarchy gives it `C_DOUT_DEFAULT_2 = 1`, so it comes up released. Channel
1 has no default, so it comes up at **0** — and it drives
`proc_sys_reset/aux_reset_in`, which is configured active-low. From power-up it
therefore holds `ap_rst_n` on all five video IPs.

An IP held in reset does not complete an AXI4-Lite transaction, and **ZynqMP has
no bus timeout on the PL ports**. So reading one of their registers does not
return an error or raise an exception — it wedges the CPU permanently. No panic,
no console output, no recovery short of pulling the power. A JTAG probe of the
same address wedges the debug port the same way.

This is why PYNQ's `Pcam5C` is handed `GPIO_IP_RESET_BaseAddress` alongside the
other three. `configure()` releases it for you, before it touches anything else;
the cell below does it explicitly so you can see it happen, and so this notebook
is safe to re-run from any cell.

In [ ]:
mipi = ol.mipi
mipi.pipeline.release_video_reset()
print("video IP reset released")

# Now these are safe to read. An HLS block reads 0x04 when it is idle and
# un-started; before the release, this loop would have hung the board.
for name in ("demosaic", "gamma_lut", "v_proc_sys", "pixel_pack"):
    print(f"  {name:<12} AP_CTRL = {getattr(mipi, name).read(0x00):#010x}")

## Bring the camera up

`ol.mipi` is the `mipi` hierarchy from `common/mipi_hier.tcl`, and which driver
class PYNQ binds to it is decided by name matching: both `Ov5647Camera` and
PYNQ's `Pcam5C` look for the same six IP — `gpio_ip_reset`,
`mipi_csi2_rx_subsyst`, `demosaic`, `gamma_lut`, `v_proc_sys` and `pixel_pack`.
Both match. The tie is broken by registration order, newest first, which is why
the import above had to come before the `Overlay`.

The cell below checks which one won, and then reads the sensor's ID register.
That read is the single most useful diagnostic in this whole notebook: it tells
you the ribbon cable is the right way round, the sensor has power, its
oscillator is running, and the I2C controller is where the device tree says.

In [ ]:
print("driver         :", type(mipi).__name__)
assert type(mipi).__name__ == "Ov5647Camera", (
    "Pcam5C won the registration race -- import ov5647 before constructing the "
    "Overlay, or restart the kernel")

print("i2c bus        :", mipi.sensor.i2c.path, "at address",
      hex(mipi.sensor.i2c.addr))
print("chip id        :", hex(mipi.sensor.chip_id()), "(0x5647 = OV5647)")

### There is no 1280x720 on this sensor

The OV5647 reads out at 2592x1944, 1920x1080, 1296x972 or 640x480, and nothing
else. CH12 is a 720p chapter — the DisplayPort mode, the accelerator's
measurements and the whole of project 2's comparison are at 720p — so
`sw/ov5647.py` derives a 720p mode rather than making the rest of the chapter
move.

It starts from the 2x2-binned readout, which averages the whole 2592x1944 array
down to 1296x972 at 87.5 Mpixel/s, and narrows the *array* window to rows
250..1705. That is 1456 rows, or 728 after binning, of which 720 are used; the
full 2624 columns give 1312 binned, of which 1280 are used. Both used areas are
exactly twice the output, so pixels stay square, and 2560x1440 array pixels is
a true 16:9 slice of a 4:3 sensor.

The frame rate then falls out of the timing grid rather than being asked for. A
line is HTS = 1896 pixel clocks at 87.5 MHz, which is 21.67 µs; VTS = 769 lines
is 16.66 ms. That is 60.0 fps — the same rate the Pcam 5C ran at, which is why
nothing downstream of here had to be re-measured.

In [ ]:
sensor_mode = mipi.configure(mode="1280x720")

print(f"sensor mode    : {sensor_mode.name}")
print(f"  window       : rows {sensor_mode.window[1]}..{sensor_mode.window[3]}"
      f" of {cam.NATIVE_HEIGHT}, {sensor_mode.binning}x binned")
print(f"  timing grid  : HTS {sensor_mode.hts} x VTS {sensor_mode.vts}"
      f" at {cam.PIXEL_RATE/1e6:.1f} Mpixel/s")
print(f"  frame rate   : {sensor_mode.fps:.2f} fps")
print(f"  MIPI link    : {cam.LANE_RATE_MBPS:.1f} Mbps on each of"
      f" {cam.MIPI_LANES} lanes")

mipi.start()
frame = mipi.readframe()
print()
print("frame          :", frame.shape, frame.dtype)
print("bytes/pixel    :", frame.shape[2])
print("physical addr  :", hex(frame.device_address))
print("pixel range    : min %d, max %d, mean %.1f"
      % (frame.min(), frame.max(), frame.mean()))

### 32 bits per pixel, not 24

`pixel_pack` in 32bpp mode appends a pad byte, giving four bytes per pixel. That
matters for everything after this project: it is what lets the accelerator —
which moves one 32-bit word per beat — read a camera frame in place, with no
repacking in software.

## Which Bayer phase? Ask the sensor

The OV5647 has no ISP worth the name. What arrives over MIPI is raw Bayer — one
colour per photosite — and the `demosaic` block in the PL reconstructs the other
two. To do that it has to be told which of the four possible arrangements the
top-left 2x2 tile is: RGGB, GRBG, GBRG or BGGR.

Getting it wrong does not produce an error. It produces a perfectly sharp,
perfectly exposed photograph in the wrong colours, and if you have never seen
the failure it reads as a white-balance problem. Worse, the answer is not
something you can look up: it depends on the sensor's mirror bit (which the
binned mode sets), on the parity of the crop offsets (which the derived 720p
mode chooses), and on how the receiver packs two pixels per clock. Three
independent things, any of which can shift the pattern by one.

So this notebook measures it, using the sensor's own colour-bar generator.
Those bars are inserted after the pixel array, so they need no lens, no light
and no correct exposure — which matters, because "the phase is wrong" and "the
room is dark" otherwise look identical.

Measured on the board this chapter was written on, the four scores were
**RGGB 72**, BGGR 2028, GRBG 2836, GBRG 3224 — RGGB by a factor of 28, and it is
what `BAYER_PHASE_DEFAULT` now says. It said `GBRG` for a while, derived from
the kernel driver's `hflip`/mbus-code table, and that was the *worst* of the
four: the picture came out magenta and green. The phase also depends on the
parity of the crop offsets this notebook's 720p mode chooses and on how the
receiver packs two pixels per clock, so reasoning from the sensor alone gets one
contribution out of three. Run the sweep on your board rather than trusting the
constant.

In [ ]:
mipi.sensor.set_test_pattern("color_bars")
time.sleep(0.5)

# OmniVision's bar order, expressed in the B,G,R order the pipeline delivers.
EXPECTED = [(255, 255, 255), (0, 255, 255), (255, 255, 0), (0, 255, 0),
            (255, 0, 255), (0, 0, 255), (255, 0, 0), (0, 0, 0)]

def band_colours(f, n=8):
    """Median colour of each of n vertical bands, from the middle of the frame."""
    h, w, _ = f.shape
    strip = f[h//2 - 20:h//2 + 20, :, :3].astype(np.int32)
    out = []
    for i in range(n):
        x0, x1 = w*i//n, w*(i+1)//n
        pad = (x1 - x0)//4          # ignore the edges, which blend
        out.append(np.median(strip[:, x0+pad:x1-pad, :].reshape(-1, 3), axis=0))
    return np.array(out, dtype=int)

scores = {}
for phase in (cam.BAYER_RGGB, cam.BAYER_GRBG, cam.BAYER_GBRG, cam.BAYER_BGGR):
    mipi.set_bayer_phase(phase)
    time.sleep(0.3)
    for _ in range(3):              # flush frames captured mid-change
        f = mipi.readframe()
    scores[phase] = int(np.abs(band_colours(f) - EXPECTED).sum())
    print(f"{cam.BAYER_NAMES[phase]}  error {scores[phase]:6d}")
    display(PIL.Image.fromarray(f[:, :, [2, 1, 0]]).resize((320, 180)))

best = min(scores, key=scores.get)
print(f"\nbest match: {cam.BAYER_NAMES[best]}"
      f"  (sw/ov5647.py defaults to {cam.BAYER_NAMES[cam.BAYER_PHASE_DEFAULT]})")
mipi.set_bayer_phase(best)

### White balance, and where it has to happen

A Bayer sensor has twice as many green photosites as red or blue, and its colour
filters are not matched to any particular illuminant. Nothing in the pipeline
corrects for that — the demosaic block interpolates, it does not balance.

**The obvious place to fix it is the wrong one.** The OV5647 has AWB gain
registers at 0x3400–0x3406, and this driver wrote them for a while. They live
inside the sensor's ISP, and in raw Bayer mode that ISP is bypassed, so they do
nothing whatsoever — measured on hardware, a requested R gain of 2.0 and B gain
of 0.5 moved the channel means by less than 0.1 counts. The give-away is that
the "after" numbers below were once byte-identical to the "before" ones.

So it is done in the **colour-space converter**, which is already sitting in the
pipeline performing an identity multiply. Scaling its diagonal is white balance
for free, and unlike the sensor's registers it actually works.

Watch the channel order, which is crossed and easy to get wrong: the frame is
B,G,R because `axis_channel_swap` put blue in the low byte, while the CSC works
in R,G,B because it sits *before* that swap.

In [ ]:
before = mipi.readframe()
print("before:  B %.1f  G %.1f  R %.1f"
      % tuple(before[:, :, c].mean() for c in range(3)))
display(PIL.Image.fromarray(before[:, :, [2, 1, 0]]).resize((320, 180)))

r, g, b = mipi.auto_white_balance()
print(f"gains :  R {r:.2f}  G {g:.2f}  B {b:.2f}")
time.sleep(0.5)

for _ in range(3):
    after = mipi.readframe()
print("after :  B %.1f  G %.1f  R %.1f"
      % tuple(after[:, :, c].mean() for c in range(3)))
display(PIL.Image.fromarray(after[:, :, [2, 1, 0]]).resize((320, 180)))

## Which byte is blue?

This is the cell the rest of the chapter rests on, so it answers the question
with a picture rather than with a claim.

AMD's `axis_channel_swap` inside the hierarchy remaps TDATA so that blue lands
in the low byte, which is why `sw/sobel_ref.py` weighs channel 0 at 29/256 and
channel 2 at 77/256, and why a frame from OpenCV — also BGR — needs no
conversion to be fed to the same filter.

If the two images below are the wrong way round, the channel swap in
`common/mipi_hier.tcl` is not what this notebook assumes, and
`HLS/src/video_filter.hpp` needs its `LUMA_*` constants reordered to match.

In [ ]:
mipi.sensor.set_test_pattern(None)
time.sleep(0.8)
for _ in range(5):
    frame = mipi.readframe()

print("interpreted as B,G,R  (what this chapter assumes) -- skin and sky should look right:")
display(PIL.Image.fromarray(frame[:, :, [2, 1, 0]]).resize((480, 270)))

print("interpreted as R,G,B  -- if THIS one looks right, the assumption is wrong:")
display(PIL.Image.fromarray(frame[:, :, :3]).resize((480, 270)))

## Out to the DisplayPort

The DisplayPort controller is hardened silicon in the PS: the DPDMA scans a
buffer out of DDR by itself, and no PL logic is involved. The Mali GPU is not
involved either — it is a renderer that could be one producer of those pixels,
and here the producer is the camera.

Allow a couple of seconds after powering the monitor on: the link trains before
`configure` will succeed.

In [ ]:
# The DisplayPort on this board offers only 24bpp modes. Asking for 32 --
# VideoMode(W, H, 32) -- is refused outright with "is not supported", and
# handing configure() an EDID mode together with PIXEL_RGB yields a
# 3-channel view over a 4-byte stride, which is not something the accelerator
# can write in place. PIXEL_RGB is DRM_FORMAT_RGB888, whose memory order is
# B,G,R -- the same order the filter and the camera use.
dp = DisplayPort()
wanted = [m for m in dp.modes if m.width == W and m.height == H]
if not wanted:
    dp.close()
    raise RuntimeError(f"monitor does not offer {W}x{H}; "
                       f"it offers {sorted({(m.width, m.height) for m in dp.modes})}")
dp_mode = max(wanted, key=lambda m: m.fps)
dp.configure(dp_mode, PIXEL_RGB)
print(f"DisplayPort {dp_mode.width}x{dp_mode.height} @ {dp_mode.fps} Hz, "
      f"{dp_mode.bits_per_pixel} bpp")

### The frame rate with nothing in the way

This is the number every later project is measured against. It is a copy loop
and nothing else: `mipi.readframe()` into a DisplayPort frame, `writeframe`,
repeat.

Note that even this copy is not free. The camera delivers 32 bits per pixel and
the DisplayPort mode is 24, so every frame is also a format conversion — and
that conversion, not the camera and not the PL, is what sets the ceiling here.
Project 2 measures it directly and finds it dominates the end-to-end rate in
every configuration, filtered or not.

In [ ]:
import cv2

N = 200
frame = dp.newframe()
t0 = time.perf_counter()
for _ in range(N):
    # BGRA -> BGR, because the DisplayPort mode is 24bpp and the camera frame
    # is 32. Doing this as `frame[:] = src[:, :, :3]` in NumPy costs about
    # 350 ms a frame -- a strided 3-of-every-4-bytes copy is the worst case for
    # the memory system. cv2.cvtColor does the same work in about 19 ms.
    cv2.cvtColor(mipi.readframe(), cv2.COLOR_BGRA2BGR, dst=frame)
    dp.writeframe(frame)
    frame = dp.newframe()
elapsed = time.perf_counter() - t0
print(f"{N} frames in {elapsed:.2f} s  =  {N/elapsed:.1f} fps")
print(f"the sensor is sending {sensor_mode.fps:.1f}")

## What the camera's own IP exposes

This section used to be an apology. The gamma LUT, the demosaic block and the
colour-space converter are all real, configurable IP sitting in the pipeline,
and PYNQ ships no driver for any of them — `libpcam5c.so` configured all four
once, at construction, and after that they were just MMIO regions with no API
in front of them.

They now have one, because replacing the blob meant writing it. Every register
offset in `sw/ov5647.py` comes from the driver headers shipped with Vivado
(`xv_demosaic_hw.h`, `xv_gamma_lut_hw.h`, `xv_csc_hw.h`), and the values that
are not simply width and height — the identity colour matrix in Q12, the `0x81`
that starts a block in auto-restart, the LUT packed two 16-bit entries to a
32-bit word — are the ones AMD's own reference design for this exact pipeline
uses, in `mipicsiss_*/examples/xmipi_ref_design/pipeline_program.c`.

**Gamma** is the one to reach for first. The sensor delivers something close to
linear light and the display expects roughly 2.2, so shadows look like black
until this is set.

In [ ]:
for gamma in (1.0, 2.2, 3.0):
    mipi.set_gamma(gamma)
    time.sleep(0.3)
    for _ in range(3):
        f = mipi.readframe()
    print(f"gamma {gamma}  (mean level {f[:, :, :3].mean():.0f})")
    display(PIL.Image.fromarray(f[:, :, [2, 1, 0]]).resize((360, 203)))

mipi.set_gamma(2.2)

### Colour, and the matrix behind it

Saturation, contrast and brightness are one 3x3 matrix and one offset vector in
the colour-space converter, and `set_color` composes all three into them. The
matrix rows are interpolated towards the BT.601 luma weights, so a fully
desaturated picture is the luminance the eye expects rather than a flat channel
average — and each row still sums to one, which is what stops "less colour"
from also meaning "less light".

Note where in the pipeline this sits: **before** `axis_channel_swap`, so the
converter still sees R,G,B. It is the swap after it that puts blue in the low
byte.

In [ ]:
for label, kwargs in [("desaturated", dict(saturation=0.0)),
                      ("normal",      dict(saturation=1.0, contrast=1.0)),
                      ("punchy",      dict(saturation=1.6, contrast=1.3))]:
    mipi.set_color(**kwargs)
    time.sleep(0.3)
    for _ in range(3):
        f = mipi.readframe()
    print(label, kwargs)
    display(PIL.Image.fromarray(f[:, :, [2, 1, 0]]).resize((360, 203)))

mipi.set_color(saturation=1.0, contrast=1.0, brightness=0)

### Exposure

The sensor's own auto-exposure loop is running — `configure()` turns it on,
which is a deliberate departure from the kernel's OV5647 driver. In Linux,
libcamera runs that loop in userspace and the driver leaves AEC/AGC in manual;
there is no libcamera here, so the choice is between letting the sensor do it
and writing the loop yourself. Left in manual with the register tables' fixed
values, the picture is correctly demosaiced and very nearly black.

Manual control is still there when you want a fixed exposure — for measuring
frame rates without the AE loop changing the answer underneath you, mostly.

In [ ]:
line_time_us = sensor_mode.hts / cam.PIXEL_RATE * 1e6
print(f"one line is {line_time_us:.2f} us, so the longest exposure this mode "
      f"allows is {(sensor_mode.vts - 4) * line_time_us / 1000:.1f} ms")

# Take the loop off the sensor first, or it simply overwrites anything set
# below on the next frame. 0x03 is manual AEC + manual AGC.
mipi.sensor.write_reg(cam.REG_AEC_AGC, 0x03)

for lines in (100, 400, 700):
    mipi.sensor.set_exposure(lines)
    time.sleep(0.3)
    for _ in range(3):
        f = mipi.readframe()
    print(f"exposure {lines:4d} lines = {lines*line_time_us/1000:5.2f} ms"
          f"   mean level {f[:, :, :3].mean():5.1f}")

# Hand it back to the sensor.
mipi.sensor.write_reg(cam.REG_AEC_AGC, 0x00)

## 1080p

The sensor's own 1080p readout: a 1:1 window out of the middle of the array, so
no binning and a narrower field of view, but full detail.

This used to be a workaround rather than a feature. `Pcam5C.__init__` called
`libpcam5c.so` once with the mode hardcoded to `MIPIMode.r1280x720_60`, and the
only way to change it was to reach past the driver and call the same library
entry point again yourself. Now it is an argument.

One number worth noticing: this mode runs at **32.8 fps**, not 30. That is what
its HTS and VTS actually give, and `sw/ov5647.py` reports the arithmetic rather
than the round number the old `MIPIMode.r1920x1080_30` name implied.

In [ ]:
dp.close()
mipi.stop()

m = mipi.configure(mode="1920x1080")
print(f"{m.name} at {m.fps:.2f} fps")
mipi.start()
f = mipi.readframe()
print("frame:", f.shape)
display(PIL.Image.fromarray(f[:, :, [2, 1, 0]]).resize((480, 270)))

# Back to 720p for anything that follows.
mipi.stop()
mipi.configure(mode="1280x720")
mipi.start()

## Clean up

Release the DisplayPort or the next run cannot open it, and stop the sensor so
it is not still driving the MIPI lanes when the PL is reprogrammed.

In [ ]:
mipi.close()
print("released")